# Notebook 03: Precipitation Indices
**ETCCDI indices from QDM bias-corrected CMIP6 data for Jakarta Greater Capital Region**

**Indices**: PRCPTOT, Rx1day, Rx3day, Rx5day, WDF, SDII  
**Models**: MRI-ESM2-0, EC-Earth3, CNRM-CM6-1  
**Scenarios**: historical, SSP1-2.6, SSP2-4.5, SSP5-8.5

> **Prerequisite:** Run `precipitation_indices.py --model all --scenario all` before executing this notebook.

## 1. Setup

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.cm as _cm
import matplotlib.pyplot as plt
import matplotlib.colors as _colors
import matplotlib.font_manager as _fm
import matplotlib.colors as _sc, matplotlib.cm as _scm
import matplotlib.lines as mlines, matplotlib.patches as mpatches

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from pathlib import Path
from scipy.interpolate import RegularGridInterpolator as _RGI

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# Paths
INDICES_DIR = Path('../../py/results/indices')
RESULTS     = Path('../../py/results/figures')
TABLES_DIR  = Path('../../py/results/tables')
RESULTS.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# CMIP6 registry
MODELS = {
    'MRI-ESM2-0': {
        'ensemble':  'r1i1p1f1',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#2364a5',
    },
    'EC-Earth3': {
        'ensemble':  'r1i1p1f1',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#e07b00',
    },
    'CNRM-CM6-1': {
        'ensemble':  'r1i1p1f2',
        'scenarios': ['historical', 'ssp126', 'ssp245', 'ssp585'],
        'color':     '#7b2d8b',
    },
}

ALL_SCENARIOS = ['historical', 'ssp126', 'ssp245', 'ssp585']
SSP_SCENARIOS = ['ssp126', 'ssp245', 'ssp585']

SCENARIO_COLORS = {
    'historical': '#676767',
    'ssp126':     '#2166AC',
    'ssp245':     '#4DAF4A',
    'ssp585':     '#D73027',
}
SCENARIO_LABELS = {
    'historical': 'Historical',
    'ssp126':     'SSP1-2.6',
    'ssp245':     'SSP2-4.5',
    'ssp585':     'SSP5-8.5',
}

HIST_PERIOD = (1950, 2014)
NEAR_PERIOD = (2021, 2050)
FAR_PERIOD  = (2071, 2100)

CENTER_LAT, CENTER_LON = -7.5, 106.875

VARIABLES = ['PRCPTOT', 'Rx1day', 'Rx3day', 'Rx5day', 'WDF', 'SDII']
UNITS_MAP  = {
    'PRCPTOT': 'mm/year',
    'Rx1day':  'mm',
    'Rx3day':  'mm',
    'Rx5day':  'mm',
    'WDF':     'days/year',
    'SDII':    'mm/wet-day',
}

print('Setup complete ✔️')
print(f'Models   : {list(MODELS.keys())}')
print(f'Scenarios: {ALL_SCENARIOS}')

In [ ]:
# Global font settings
PLOT_FONT        = 'Calibri'                     # preferred font
PLOT_FONT_ALT    = ['Arial', 'Times New Roman']  # fallback chain

_available = {f.name for f in _fm.fontManager.ttflist}
_font_to_use = PLOT_FONT if PLOT_FONT in _available else next(
    (f for f in PLOT_FONT_ALT if f in _available), 'Times New Roman'
)

mpl.rcParams['font.family'] = _font_to_use
mpl.rcParams['axes.titlesize']  = 11
mpl.rcParams['axes.labelsize']  = 10
mpl.rcParams['xtick.labelsize'] = 9
mpl.rcParams['ytick.labelsize'] = 9
mpl.rcParams['legend.fontsize'] = 9

print(f'Font set to: {_font_to_use}')
if _font_to_use != PLOT_FONT:
    print(f'  (Note: {PLOT_FONT!r} not found — using {_font_to_use!r} instead)')

### Configs

In [ ]:
def load_indices(model, scenario, indices_dir=INDICES_DIR):
    """
    Load pre-computed ETCCDI indices for one model × scenario.
    Returns Dataset or None if file is missing.
    Input filename: {model}_{scenario}_{ensemble}_indices_jakarta.nc
    """
    cfg      = MODELS[model]
    ensemble = cfg['ensemble']
    fname    = f'{model}_{scenario}_{ensemble}_indices_jakarta.nc'
    fpath    = indices_dir / fname
    if not fpath.exists():
        return None
    return xr.open_dataset(
        fpath,
        decode_times=xr.coders.CFDatetimeCoder(use_cftime=True)
    )




def plot_spatial(da, title, cmap, units, output_path=None, diverging=False):
    """
    Plot a 2-D (lat × lon) DataArray as a smooth Cartopy map.

    Upsamples the coarse CMIP6 grid to 300×300 using RegularGridInterpolator
    before rendering, producing smooth continuous gradients. Uses imshow with
    the Cartopy transform for clean axis alignment.
    """

    # Display parameters
    MAP_FONT        = 'Calibri'      # preferred font
    MAP_FONT_ALT    = ['Arial', 'Times New Roman']  # fallback chain
    FIG_WIDTH       = 9              # figure width in inches
    FIG_HEIGHT      = 5              # figure height in inches
    UPSAMPLE_N      = 300            # interpolation grid resolution (N × N)
    COASTLINE_WIDTH = 1.0
    BORDER_WIDTH    = 0.4
    GRIDLINE_ALPHA  = 0.5
    COLORBAR_SHRINK = 0.85
    TITLE_FONTSIZE  = 11
    DPI             = 150
    OCEAN_COLOR     = '#d0e8f0'

    # Apply font preference
    _available = {f.name for f in _fm.fontManager.ttflist}
    _font = MAP_FONT if MAP_FONT in _available else next(
        (f for f in MAP_FONT_ALT if f in _available), 'Times New Roman'
    )
    plt.rcParams['font.family'] = _font

    lons = da.lon.values
    lats = da.lat.values
    vals = da.values.astype(float)

    # Colour scale from coarse data
    finite = vals[np.isfinite(vals)]
    if diverging:
        vmax = float(np.abs(finite).max()) if len(finite) else 1.0
        vmin_plot, vmax_plot = -vmax, vmax
    else:
        vmin_plot = float(finite.min()) if len(finite) else 0.0
        vmax_plot = float(finite.max()) if len(finite) else 1.0

    # Upsample: RegularGridInterpolator requires ascending lat axis
    lat_asc   = lats if lats[0] < lats[-1] else lats[::-1]
    vals_asc  = vals if lats[0] < lats[-1] else vals[::-1, :]

    interp = _RGI(
        (lat_asc, lons), vals_asc,
        method='linear',
        bounds_error=False,
        fill_value=np.nan,
    )
    lons_fine = np.linspace(lons.min(), lons.max(), UPSAMPLE_N)
    lats_fine = np.linspace(lat_asc.min(), lat_asc.max(), UPSAMPLE_N)
    grid_lon, grid_lat = np.meshgrid(lons_fine, lats_fine)
    vals_up = interp((grid_lat, grid_lon))

    fig, ax = plt.subplots(
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )

    # OCEAN background
    ax.add_feature(cfeature.OCEAN, facecolor=OCEAN_COLOR, zorder=2)

    # Data: imshow on the upsampled array with bicubic smoothing on top
    ax.imshow(
        vals_up,
        extent=[lons.min(), lons.max(), lat_asc.min(), lat_asc.max()],
        origin='lower',
        cmap=cmap,
        vmin=vmin_plot, vmax=vmax_plot,
        interpolation='bicubic',
        transform=ccrs.PlateCarree(),
        zorder=1,
        aspect='auto',
    )

    # Dummy pcolormesh for the colorbar (imshow colorbar works but pcolormesh
    # gives a cleaner scalar mappable with the exact same vmin/vmax)
    _norm = _colors.Normalize(vmin=vmin_plot, vmax=vmax_plot)
    _sm   = plt.cm.ScalarMappable(cmap=cmap, norm=_norm)
    _sm.set_array([])
    plt.colorbar(_sm, ax=ax, label=units, shrink=COLORBAR_SHRINK, pad=0.02)

    ax.add_feature(cfeature.COASTLINE, linewidth=COASTLINE_WIDTH, zorder=5)
    ax.add_feature(cfeature.BORDERS,   linewidth=BORDER_WIDTH, linestyle=':', zorder=4)

    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color='gray',
                      alpha=GRIDLINE_ALPHA, linestyle='--')
    gl.top_labels   = False
    gl.right_labels = False

    ax.set_extent([104.125, 109.375, -7.5, -5], crs=ccrs.PlateCarree())
    ax.set_title(title, fontsize=TITLE_FONTSIZE, fontweight='bold')
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=DPI, bbox_inches='tight')
    return fig

print('Helpers defined ✔️')

## 2. File Inventory

If files are missing, run:
```
python py/indices/precipitation_indices.py --model all --scenario all
```

In [ ]:
print('Indices file inventory:')
print('=' * 70)

all_ds = {}   # {(model, scenario): Dataset}

for model, cfg in MODELS.items():
    ensemble = cfg['ensemble']
    for scenario in cfg['scenarios']:
        ds = load_indices(model, scenario)
        fname = f'{model}_{scenario}_{ensemble}_indices_jakarta.nc'
        if ds is None:
            print(f'  ❌ MISSING  {model:<15} {scenario:<12} {fname}')
        else:
            years = ds['year'].values
            print(f'  ✔️          {model:<15} {scenario:<12} years={int(years[0])}–{int(years[-1])}  ({len(years)} yrs)')
            all_ds[(model, scenario)] = ds

print('=' * 70)
print(f'Loaded: {len(all_ds)} / {sum(len(cfg["scenarios"]) for cfg in MODELS.values())} datasets')

## 3. Spatial Maps based on Historical Mean

One row per model, one column per index. Shows Jakarta's spatial precipitation gradients as seen through QDM-corrected CMIP6 data.

In [ ]:
INDEX_CONFIGS = [
    ('PRCPTOT', 'Blues',   'mm/year',    'PRCPTOT (Annual Total Wet-Day Precipitation)'),
    ('Rx1day',  'YlOrRd',  'mm',         'Rx1day (Annual Max 1-Day Precipitation)'),
    ('Rx5day',  'YlOrRd',  'mm',         'Rx5day (Annual Max 5-Day Precipitation)'),
    ('WDF',     'YlGnBu',  'days/year',  'WDF (Wet-Day Frequency)'),
    ('SDII',    'PuRd',    'mm/wet-day', 'SDII (Simple Daily Intensity Index)'),
]

for model in MODELS:
    ds_hist = all_ds.get((model, 'historical'))
    if ds_hist is None:
        print(f'⚠ Skipping {model} — historical indices not loaded')
        continue

    # Slice to HIST_PERIOD for the mean
    years     = ds_hist['year'].values
    hist_mask = (years >= HIST_PERIOD[0]) & (years <= HIST_PERIOD[1])
    ds_mean   = ds_hist.sel(year=years[hist_mask]).mean(dim='year')

    print(f'\n--- {model} (Historical mean {HIST_PERIOD[0]}–{HIST_PERIOD[1]}) ---')
    for var, cmap, units, var_title in INDEX_CONFIGS:
        if var not in ds_mean:
            continue
        fig = plot_spatial(
            ds_mean[var],
            title=f'{var_title}\n Historical Mean ({HIST_PERIOD[0]}–{HIST_PERIOD[1]}) of {model}',
            cmap=cmap, units=units,
            output_path=RESULTS / f'03_{var}_{model}_historical_mean.png',
        )
        plt.show()

## 4. Time Series at Jakarta Centre Cell

One subplot per index, all models × scenarios overlaid with a 10-year rolling mean.  
Models are distinguished by linestyle; scenarios by colour.

In [ ]:
# Time series display parameters
TS_FIG_WIDTH        = 14       # figure width in inches
TS_FIG_HEIGHT       = 5        # figure height per panel (multiplied by n_panels)
TS_INDIV_ALPHA      = 0.25     # individual model line transparency
TS_INDIV_WIDTH      = 0.8      # individual model line width
TS_ENS_WIDTH        = 2.5      # ensemble mean line width
TS_ROLL_WINDOW      = 10       # rolling mean window (years)
TS_BAND_ALPHA       = 0.20     # uncertainty band (±1σ) transparency
TS_GLOREDA_STYLE    = dict(color='#1a1a1a', linewidth=1.2, linestyle='dotted', alpha=0.7)
TS_VLINE_STYLE      = dict(color='gray', linewidth=0.9, linestyle=':', alpha=0.5)
TS_NEAR_SPAN_ALPHA  = 0.06     # near-term shading transparency
TS_FAR_SPAN_ALPHA   = 0.10     # far-term shading transparency
TS_GRID_ALPHA       = 0.25     # background grid transparency
TS_LEGEND_FONTSIZE  = 9
TS_LABEL_FONTSIZE   = 11
TS_TITLE_FONTSIZE   = 14
TS_DPI              = 150

MODEL_LINESTYLES = {'MRI-ESM2-0': '-', 'EC-Earth3': '--', 'CNRM-CM6-1': ':'}

fig, axes = plt.subplots(3, 2, figsize=(TS_FIG_WIDTH, TS_FIG_HEIGHT * 3))
axes = axes.flatten()

for ax, var in zip(axes, VARIABLES):
    for model in MODELS:
        ls = MODEL_LINESTYLES[model]
        for scenario in ALL_SCENARIOS:
            ds = all_ds.get((model, scenario))
            if ds is None or var not in ds:
                continue
            da    = ds[var].sel(lat=CENTER_LAT, lon=CENTER_LON, method='nearest')
            years = da.year.values.astype(int)
            vals  = da.values.astype(float)
            roll  = pd.Series(vals, index=years).rolling(TS_ROLL_WINDOW, center=True).mean()

            ax.plot(years, vals,  alpha=TS_INDIV_ALPHA,
                    color=SCENARIO_COLORS[scenario], linewidth=TS_INDIV_WIDTH,
                    linestyle=ls)
            ax.plot(years, roll.values, linestyle=ls, linewidth=TS_ENS_WIDTH * 0.7,
                    color=SCENARIO_COLORS[scenario])

    ax.axvline(2015, **TS_VLINE_STYLE)
    ax.axvspan(*NEAR_PERIOD, alpha=TS_NEAR_SPAN_ALPHA, color='orange')
    ax.axvspan(*FAR_PERIOD,  alpha=TS_FAR_SPAN_ALPHA,  color='red')
    ax.set_title(f'{var}  [{UNITS_MAP[var]}]', fontweight='bold', fontsize=10)
    ax.set_ylabel(UNITS_MAP[var], fontsize=9)
    ax.set_xlabel('Year', fontsize=9)
    ax.grid(True, alpha=TS_GRID_ALPHA)

# Shared legend
scen_patches = [mpatches.Patch(color=SCENARIO_COLORS[s], label=SCENARIO_LABELS[s])
                for s in ALL_SCENARIOS]
model_lines  = [mlines.Line2D([], [], color='grey', linestyle=MODEL_LINESTYLES[m], label=m)
                for m in MODELS]
axes[0].legend(handles=scen_patches + model_lines, fontsize=TS_LEGEND_FONTSIZE,
               loc='upper left', ncol=2)

plt.suptitle('ETCCDI Precipitation Indices at the Jakarta Centre Cell\n'
             'All CMIP6 models × scenarios  |  10-year rolling mean\n'
             '(Solid = MRI-ESM2-0  Dashed = EC-Earth3  Dotted = CNRM-CM6-1)',
             fontsize=TS_TITLE_FONTSIZE, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / '03_indices_timeseries_all.png', dpi=TS_DPI, bbox_inches='tight')
plt.show()

## 5. Change Signal Maps: Far Future vs Historical

Relative change (%) per index per SSP scenario, spatially averaged across all three models.  
Rows = indices, columns = SSP scenarios.

In [ ]:
_av2 = {f.name for f in _fm.fontManager.ttflist}

mpl.rcParams['font.family'] = PLOT_FONT if PLOT_FONT in _av2 else next(
    (f for f in ['Calibri', 'Arial', 'Times New Roman'] if f in _av2), 'Times New Roman')

CHANGE_VARS = ['PRCPTOT', 'Rx1day', 'Rx3day', 'Rx5day', 'SDII', 'WDF']

for var in CHANGE_VARS:
    fig, axes = plt.subplots(
        1, len(SSP_SCENARIOS), figsize=(5 * len(SSP_SCENARIOS), 4),
        subplot_kw={'projection': ccrs.PlateCarree()}
    )

    for ax, ssp in zip(axes, SSP_SCENARIOS):
        deltas   = []
        lons_ref = lats_ref = None

        for model in MODELS:
            ds_hist = all_ds.get((model, 'historical'))
            ds_fut  = all_ds.get((model, ssp))
            if ds_hist is None or ds_fut is None or var not in ds_hist:
                continue
            hist_yrs = ds_hist['year'].values
            fut_yrs  = ds_fut['year'].values
            hist_mean = ds_hist[var].sel(
                year=hist_yrs[(hist_yrs >= HIST_PERIOD[0]) & (hist_yrs <= HIST_PERIOD[1])]
            ).mean(dim='year')
            far_mask = (fut_yrs >= FAR_PERIOD[0]) & (fut_yrs <= FAR_PERIOD[1])
            if far_mask.sum() == 0:
                continue
            fut_mean   = ds_fut[var].sel(year=fut_yrs[far_mask]).mean(dim='year')
            delta_vals = ((fut_mean - hist_mean) / hist_mean.where(hist_mean != 0)) * 100
            m_lats = ds_hist.lat.values
            m_lons = ds_hist.lon.values
            d_vals = delta_vals.values.astype(float)
            if lons_ref is None:
                lons_ref = m_lons; lats_ref = m_lats
                deltas.append(d_vals)
            elif d_vals.shape == deltas[0].shape:
                deltas.append(d_vals)
            else:
                lat_asc = m_lats if m_lats[0] < m_lats[-1] else m_lats[::-1]
                val_asc = d_vals if m_lats[0] < m_lats[-1] else d_vals[::-1, :]
                interp  = _RGI((lat_asc, m_lons), val_asc,
                               method='linear', bounds_error=False, fill_value=np.nan)
                ref_lat_asc = lats_ref if lats_ref[0] < lats_ref[-1] else lats_ref[::-1]
                gg_lon, gg_lat = np.meshgrid(lons_ref, ref_lat_asc)
                remapped = interp((gg_lat, gg_lon))
                if lats_ref[0] > lats_ref[-1]:
                    remapped = remapped[::-1, :]
                deltas.append(remapped)

        if not deltas:
            ax.set_visible(False)
            continue

        ens_delta = np.nanmean(np.stack(deltas, axis=0), axis=0)
        vmax      = float(np.abs(ens_delta[np.isfinite(ens_delta)]).max())

        # Smooth RGI + imshow rendering
        _lat_asc  = lats_ref if lats_ref[0] < lats_ref[-1] else lats_ref[::-1]
        _val_asc  = ens_delta if lats_ref[0] < lats_ref[-1] else ens_delta[::-1, :]
        _interp   = _RGI((_lat_asc, lons_ref), _val_asc,
                          method='linear', bounds_error=False, fill_value=np.nan)
        _lf  = np.linspace(lons_ref.min(), lons_ref.max(), 300)
        _ltf = np.linspace(_lat_asc.min(), _lat_asc.max(), 300)
        _gl, _glt = np.meshgrid(_lf, _ltf)
        _up = _interp((_glt, _gl))

        ax.add_feature(cfeature.OCEAN, facecolor='#d0e8f0', zorder=2)
        ax.imshow(_up,
                  extent=[lons_ref.min(), lons_ref.max(), _lat_asc.min(), _lat_asc.max()],
                  origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  interpolation='bicubic',
                  transform=ccrs.PlateCarree(), zorder=1, aspect='auto')
        _norm = _sc.Normalize(vmin=-vmax, vmax=vmax)
        _sm   = _scm.ScalarMappable(cmap='RdBu_r', norm=_norm)
        _sm.set_array([])
        plt.colorbar(_sm, ax=ax, label='%', shrink=0.85)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=5)
        ax.set_extent([104.125, 109.375, -7.5, -5], crs=ccrs.PlateCarree())
        ax.set_title(SCENARIO_LABELS[ssp], fontweight='bold')

    fig.suptitle(f'Relative Change of {var}: Far Future ({FAR_PERIOD[0]}–{FAR_PERIOD[1]}) vs '
                 f'Historical ({HIST_PERIOD[0]}–{HIST_PERIOD[1]})\n'
                 f'3-model ensemble mean  |  Red colours = increase, blue = decrease',
                 fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig(RESULTS / f'03_{var}_change_signal_ensemble.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Per-Model Change Signal at Centre Cell

Bar chart showing % change for each model separately, for PRCPTOT and Rx1day.  
Reveals inter-model spread in the projected signal.

In [ ]:
BAR_VARS = ['PRCPTOT', 'Rx1day', 'Rx3day', 'Rx5day', 'SDII', 'WDF']

fig, axes = plt.subplots(1, len(BAR_VARS), figsize=(3 * len(BAR_VARS), 4))

model_list  = list(MODELS.keys())
n_models    = len(model_list)
bar_width   = 0.25
x           = np.arange(len(SSP_SCENARIOS))

for ax, var in zip(axes, BAR_VARS):
    for i, model in enumerate(model_list):
        ds_hist = all_ds.get((model, 'historical'))
        if ds_hist is None:
            continue

        hist_yrs  = ds_hist['year'].values
        hist_mask = (hist_yrs >= HIST_PERIOD[0]) & (hist_yrs <= HIST_PERIOD[1])
        hist_val  = float(
            ds_hist[var].sel(year=hist_yrs[hist_mask]).mean(dim='year')
            .sel(lat=CENTER_LAT, lon=CENTER_LON, method='nearest')
        )

        pct_changes = []
        for ssp in SSP_SCENARIOS:
            ds_fut = all_ds.get((model, ssp))
            if ds_fut is None:
                pct_changes.append(np.nan)
                continue
            fut_yrs  = ds_fut['year'].values
            far_mask = (fut_yrs >= FAR_PERIOD[0]) & (fut_yrs <= FAR_PERIOD[1])
            if far_mask.sum() == 0:
                pct_changes.append(np.nan)
                continue
            fut_val = float(
                ds_fut[var].sel(year=fut_yrs[far_mask]).mean(dim='year')
                .sel(lat=CENTER_LAT, lon=CENTER_LON, method='nearest')
            )
            pct_changes.append((fut_val - hist_val) / hist_val * 100 if hist_val != 0 else np.nan)

        offset = (i - (n_models - 1) / 2) * bar_width
        ax.bar(x + offset, pct_changes, width=bar_width,
               color=MODELS[model]['color'], alpha=0.85,
               label=model, edgecolor='white')

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([SCENARIO_LABELS[s] for s in SSP_SCENARIOS], fontsize=9)
    ax.set_ylabel('% Change vs Historical')
    ax.set_title(f'{var}', fontweight='bold')
    ax.grid(True, axis='y', alpha=0.4)

axes[0].legend(fontsize=8)
plt.suptitle(f'Index % Change: Far Future ({FAR_PERIOD[0]}–{FAR_PERIOD[1]}) vs '
             f'Historical ({HIST_PERIOD[0]}–{HIST_PERIOD[1]})\nJakarta Centre Cell',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS / '03_indices_change_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Period-Mean Summary Tables

Spatially averaged period means for each model × scenario × period.  
Saved to `py/results/tables/`.

In [ ]:
period_defs = [
    ('Historical',  HIST_PERIOD, 'historical'),
    ('Near-term',   NEAR_PERIOD, None),   # None -> SSPs only
    ('Far-term',    FAR_PERIOD,  None),
]

rows = []
for model in MODELS:
    for period_label, (yr0, yr1), fixed_scen in period_defs:
        scens = [fixed_scen] if fixed_scen else SSP_SCENARIOS
        for scen in scens:
            ds = all_ds.get((model, scen))
            if ds is None:
                continue
            yrs  = ds['year'].values
            mask = (yrs >= yr0) & (yrs <= yr1)
            if mask.sum() == 0:
                continue
            ds_p = ds.sel(year=yrs[mask]).mean(dim='year')
            row  = {
                'Model':    model,
                'Scenario': SCENARIO_LABELS.get(scen, scen),
                'Period':   f'{period_label} ({yr0}–{yr1})',
            }
            for var in VARIABLES:
                if var in ds_p:
                    row[var] = round(float(ds_p[var].mean()), 1)
            rows.append(row)

summary_df = pd.DataFrame(rows)
print('Spatially-Averaged Period Mean Indices:')
print(summary_df.to_string(index=False))
summary_df.to_csv(TABLES_DIR / 'indices_period_means.csv', index=False)
print(f'\nSaved → py/results/tables/indices_period_means.csv')

In [ ]:
# % change table: far-future vs historical, per model × SSP
change_rows = []
for model in MODELS:
    ds_hist = all_ds.get((model, 'historical'))
    if ds_hist is None:
        continue
    hist_yrs  = ds_hist['year'].values
    hist_mask = (hist_yrs >= HIST_PERIOD[0]) & (hist_yrs <= HIST_PERIOD[1])
    ds_h_mean = ds_hist.sel(year=hist_yrs[hist_mask]).mean(dim='year')

    for ssp in SSP_SCENARIOS:
        ds_fut = all_ds.get((model, ssp))
        if ds_fut is None:
            continue
        fut_yrs  = ds_fut['year'].values
        far_mask = (fut_yrs >= FAR_PERIOD[0]) & (fut_yrs <= FAR_PERIOD[1])
        if far_mask.sum() == 0:
            continue
        ds_f_mean = ds_fut.sel(year=fut_yrs[far_mask]).mean(dim='year')

        row = {'Model': model, 'Scenario': SCENARIO_LABELS[ssp]}
        for var in VARIABLES:
            if var not in ds_h_mean:
                continue
            h_val = float(ds_h_mean[var].mean())
            f_val = float(ds_f_mean[var].mean())
            row[f'{var} Δ%'] = round((f_val - h_val) / h_val * 100, 1) if h_val != 0 else np.nan
        change_rows.append(row)

change_df = pd.DataFrame(change_rows)
print(f'% Change: Far Future ({FAR_PERIOD[0]}–{FAR_PERIOD[1]}) vs Historical ({HIST_PERIOD[0]}–{HIST_PERIOD[1]})')
print(change_df.to_string(index=False))
change_df.to_csv(TABLES_DIR / 'indices_pct_change.csv', index=False)
print(f'\nSaved → py/results/tables/indices_pct_change.csv')

## ✅ Summary

**Indices computed by `precipitation_indices.py`:**
- PRCPTOT, Rx1day, Rx3day, Rx5day, WDF, SDII for all 3 models × 4 scenarios
- Year-loop groupby (never `.resample()`), which is safe for all cftime calendars
- Wet-day threshold: ≥1 mm/day

**Key findings:**
- All models agree on the sign of change: PRCPTOT and Rx1day increase under all SSPs
- SSP5-8.5 shows the largest far-future signal across all indices
- Inter-model spread (bar chart) is largest for Rx1day, therefore relevant uncertainty for the erosivity projections
- The spatial change signal is relatively uniform across the small Jakarta domain

**Next:** `04_water_stress.ipynb`